In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import BooleanType
from pyspark.sql.window import Window

In [0]:
SILVER_FUEL_TABLE  = "us_grid_energy_pipeline_databricks.usgrid.silver_fuel_type_data"
DIM_FUEL_TABLE     = "us_grid_energy_pipeline_databricks.usgrid.dim_fuel_type"
DIM_REGION_TABLE   = "us_grid_energy_pipeline_databricks.usgrid.dim_region"

GOLD_PATH  = "abfss://gold@usgridenergypipeline.dfs.core.windows.net/gold_fuel_metrics"
GOLD_TABLE = "us_grid_energy_pipeline_databricks.usgrid.gold_fuel_metrics"

RENEWABLE_FUELS = ["SUN", "WND", "WAT", "GEO", "NUC"]

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS us_grid_energy_pipeline_databricks.usgrid")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
        region_code     STRING,
        region_name     STRING,
        period          TIMESTAMP,
        fuel_type_code  STRING,
        fuel_type_name  STRING,
        value_mw        FLOAT,
        percentage_of_total    FLOAT,
        is_renewable    BOOLEAN
    )
    USING DELTA
    LOCATION '{GOLD_PATH}'
    PARTITIONED BY (region_code, fuel_type_code)
""")

DataFrame[]

In [0]:
df_silver = spark.read.table(SILVER_FUEL_TABLE)
df_dim_fuel = spark.read.table(DIM_FUEL_TABLE)
df_dim_region = spark.read.table(DIM_REGION_TABLE)

In [0]:
df_joined = (
    df_silver
    .join(df_dim_fuel, on="fuel_type_code", how="left")
    .join(df_dim_region, on="region_code", how="left")
)

In [0]:
df_metrics = (
    df_joined
    .withColumn(
        "percentage_of_total",
        F.round(
            F.when(F.sum("value_in_megawatts").over(window) != 0,
                F.col("value_in_megawatts") / F.sum("value_in_megawatts").over(window) * 100
            ).otherwise(0),
        2)
    )
    .withColumn(
        "is_renewable",
        F.col("fuel_type_code").isin(RENEWABLE_FUELS)
    )
    .withColumnRenamed("value_in_megawatts", "value_mw")
)

In [0]:
df_gold = df_metrics.select(
    "region_code",
    "region_name",
    "period",
    "fuel_type_code",
    "fuel_type_name",
    "value_mw",
    "percentage_of_total",
    "is_renewable"
)

In [0]:
df_gold.createOrReplaceTempView("gold_updates")

spark.sql(f"""
    MERGE INTO {GOLD_TABLE} AS target
    USING gold_updates AS source
    ON target.region_code = source.region_code
    AND target.fuel_type_code = source.fuel_type_code
    AND target.period = source.period
    WHEN MATCHED AND (
        target.value_mw != source.value_mw OR
        target.percentage_of_total != source.percentage_of_total
    )
        THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql(f"SELECT * FROM {GOLD_TABLE} LIMIT 10").display()

region_code,region_name,period,fuel_type_code,fuel_type_name,value_mw,percentage_of_total,is_renewable
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-01T03:00:00.000Z,WND,Wind,20633.0,29.17,true
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-01T04:00:00.000Z,WND,Wind,20096.0,29.09,true
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-01T07:00:00.000Z,WND,Wind,19965.0,31.43,true
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-01T21:00:00.000Z,WND,Wind,15258.0,22.45,true
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-01T23:00:00.000Z,WND,Wind,11277.0,15.94,true
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-02T08:00:00.000Z,WND,Wind,5965.0,8.96,true
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-02T10:00:00.000Z,WND,Wind,5331.0,7.94,true
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-02T20:00:00.000Z,WND,Wind,7516.0,9.93,true
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-02T22:00:00.000Z,WND,Wind,9439.0,12.37,true
MISO,"Midcontinent Independent System Operator, Inc.",2025-01-03T05:00:00.000Z,WND,Wind,15008.0,19.9,true


In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {GOLD_TABLE}")
# dbutils.fs.rm("abfss://gold-dev@usgridenergypipeline.dfs.core.windows.net/gold_fuel_metrics", recurse=True)